# AI Virtual Assistant Brev Launchable

Use this notebook to deploy the no-GPU Brev lab version of the AI Virtual Assistant. It uses NVIDIA-hosted NIMs, public GHCR app images, and CPU Milvus.

Run each cell from top to bottom. The first cell prompts for your NVIDIA API key. The deploy cell pulls prebuilt images and starts Docker Compose with `--no-build`, so this should be much faster than building containers locally.

After deployment, open the Brev secure link for port `3001` to use the sample UI, then run `notebooks/ingest_data.ipynb` to load sample data.


In [ ]:
import getpass
import os

USE_GHCR_IMAGES = os.environ.get("USE_GHCR_IMAGES", "1").strip().lower() not in {"0", "false", "no"}
GHCR_OWNER = os.environ.get("GHCR_OWNER", "jspaulding-nv")
GHCR_IMAGE_PREFIX = os.environ.get("GHCR_IMAGE_PREFIX", "aiva-customer-service")
GHCR_TAG = os.environ.get("GHCR_TAG", "nemotron3-milvus-cpu")

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("Enter your NVIDIA API key: ")

if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY is required.")

NGC_API_KEY = os.environ.get("NGC_API_KEY", "")
if not USE_GHCR_IMAGES and not NGC_API_KEY:
    NGC_API_KEY = getpass.getpass("Docker/NGC API key (press Enter to reuse NVIDIA_API_KEY): ") or NVIDIA_API_KEY

if not USE_GHCR_IMAGES and not NGC_API_KEY:
    raise ValueError("NGC_API_KEY is required when building images locally from source.")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if NGC_API_KEY:
    os.environ["NGC_API_KEY"] = NGC_API_KEY
os.environ["USE_GHCR_IMAGES"] = "1" if USE_GHCR_IMAGES else "0"
os.environ["GHCR_OWNER"] = GHCR_OWNER
os.environ["GHCR_IMAGE_PREFIX"] = GHCR_IMAGE_PREFIX
os.environ["GHCR_TAG"] = GHCR_TAG

print("NVIDIA_API_KEY is set.")
if USE_GHCR_IMAGES:
    print(f"Using public GHCR app images: ghcr.io/{GHCR_OWNER}/{GHCR_IMAGE_PREFIX}-*:{GHCR_TAG}")
else:
    print("Using local source builds. NGC_API_KEY is set for nvcr.io pulls.")

In [ ]:
from pathlib import Path
import os


def find_repo_root():
    start = Path.cwd().resolve()
    for path in (start, *start.parents):
        if (path / "deploy" / "compose" / "docker-compose.yaml").exists():
            return path

    for candidate in [Path.home() / "ai-virtual-assistant", Path("/home/ubuntu/ai-virtual-assistant")]:
        if (candidate / "deploy" / "compose" / "docker-compose.yaml").exists():
            return candidate

    raise FileNotFoundError("Could not find deploy/compose/docker-compose.yaml. Open this notebook from the ai-virtual-assistant repo.")


REPO_ROOT = find_repo_root()
COMPOSE_FILE = REPO_ROOT / "deploy" / "compose" / "docker-compose.yaml"
GHCR_COMPOSE_FILE = REPO_ROOT / "deploy" / "compose" / "docker-compose.ghcr.yaml"
ENV_FILE = REPO_ROOT / ".env.launchable"
os.chdir(REPO_ROOT)

COMPOSE_FILES = [COMPOSE_FILE]
if USE_GHCR_IMAGES:
    if not GHCR_COMPOSE_FILE.exists():
        raise FileNotFoundError(f"GHCR Compose override not found: {GHCR_COMPOSE_FILE}")
    COMPOSE_FILES.append(GHCR_COMPOSE_FILE)

COMPOSE_ARGS = []
for compose_file in COMPOSE_FILES:
    COMPOSE_ARGS.extend(["-f", str(compose_file)])

print(f"Repository root: {REPO_ROOT}")
print("Compose files:")
for compose_file in COMPOSE_FILES:
    print(f"- {compose_file}")

In [ ]:
env_lines = [
    f"NVIDIA_API_KEY={NVIDIA_API_KEY}",
    f"NGC_API_KEY={NGC_API_KEY}",
    "APP_LLM_MODELNAME=nvidia/nemotron-3-nano-30b-a3b",
    "APP_VECTORSTORE_INDEXTYPE=IVF_FLAT",
    f"USE_GHCR_IMAGES={'1' if USE_GHCR_IMAGES else '0'}",
    f"GHCR_OWNER={GHCR_OWNER}",
    f"GHCR_IMAGE_PREFIX={GHCR_IMAGE_PREFIX}",
    f"GHCR_TAG={GHCR_TAG}",
    "",
]
ENV_FILE.write_text("\n".join(env_lines), encoding="utf-8")
ENV_FILE.chmod(0o600)

print(f"Wrote environment file: {ENV_FILE}")

In [ ]:
import subprocess
import time
from collections import deque

LOG_DIR = REPO_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
DEPLOY_LOG = LOG_DIR / "deploy_hosted_nims.log"


def docker_cmd(*args):
    probe = subprocess.run(["docker", "info"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if probe.returncode == 0:
        return ["docker", *args]
    return ["sudo", "docker", *args]


def run_logged(command, log_file, error_message):
    recent_lines = deque(maxlen=8)

    print(f"Running: {' '.join(str(part) for part in command)}", flush=True)
    print(f"Streaming Docker output to {log_file}", flush=True)
    last_progress = time.monotonic()

    with log_file.open("a", encoding="utf-8") as log:
        log.write(f"\n\n$ {' '.join(str(part) for part in command)}\n")
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.strip()
            if stripped:
                recent_lines.append(stripped)
            now = time.monotonic()
            if now - last_progress >= 10:
                print(".", end="", flush=True)
                last_progress = now

        return_code = process.wait()

    print(" done", flush=True)

    if return_code != 0:
        print(error_message)
        print(f"Full Docker output: {log_file}")
        print("Last Docker output lines:")
        for line in recent_lines:
            print(line)
        raise RuntimeError(f"Command failed with exit code {return_code}.")

    return return_code


version = subprocess.run(docker_cmd("compose", "version"), text=True, capture_output=True, check=True)
print(version.stdout.strip())

if USE_GHCR_IMAGES:
    GHCR_USER = os.environ.get("GHCR_USER", "")
    GHCR_TOKEN = os.environ.get("GHCR_TOKEN", "")
    if GHCR_USER and GHCR_TOKEN:
        login = subprocess.run(
            docker_cmd("login", "ghcr.io", "-u", GHCR_USER, "--password-stdin"),
            input=GHCR_TOKEN,
            text=True,
            capture_output=True,
        )
        if login.returncode != 0:
            print(login.stdout)
            print(login.stderr)
            raise RuntimeError("Docker login to ghcr.io failed.")
        print("Docker is ready and authenticated with ghcr.io.")
    else:
        print("Docker is ready. GHCR images are public, so no GHCR login is required.")
else:
    login = subprocess.run(
        docker_cmd("login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"),
        input=NGC_API_KEY,
        text=True,
        capture_output=True,
    )
    if login.returncode != 0:
        print(login.stdout)
        print(login.stderr)
        raise RuntimeError("Docker login to nvcr.io failed. If you used a Build API key, use an NGC personal key for NGC_API_KEY.")

    print("Docker is ready and authenticated with nvcr.io.")

In [ ]:
config = subprocess.run(
    docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "config"),
    text=True,
    capture_output=True,
)
if config.returncode != 0:
    print(config.stdout)
    print(config.stderr)
    raise RuntimeError("Docker Compose config validation failed.")

rendered = config.stdout
assert "nvidia/nemotron-3-nano-30b-a3b" in rendered
assert "milvusdb/milvus:v2.4.15" in rendered
assert "v2.4.15-gpu" not in rendered
assert "KNOWHERE_GPU_MEM_POOL_SIZE" not in rendered

if USE_GHCR_IMAGES:
    assert f"ghcr.io/{GHCR_OWNER}/{GHCR_IMAGE_PREFIX}-agent:{GHCR_TAG}" in rendered
    assert "\nbuild:" not in rendered
    print("Docker Compose configuration validated for hosted Nemotron 3 Nano, CPU Milvus, and GHCR app images.")
else:
    print("Docker Compose configuration validated for hosted Nemotron 3 Nano and CPU Milvus.")

In [ ]:
DEPLOY_LOG.write_text("", encoding="utf-8")

if USE_GHCR_IMAGES:
    print("Pulling public GHCR app images and upstream service images. This can take a few minutes on a fresh VM.", flush=True)
    run_logged(
        docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "pull"),
        DEPLOY_LOG,
        "Docker Compose image pull failed.",
    )
    print("Starting Docker Compose services without local builds.", flush=True)
    run_logged(
        docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "up", "-d", "--no-build"),
        DEPLOY_LOG,
        "Docker Compose deployment failed.",
    )
else:
    print("Building and starting Docker Compose services from local source. This can take several minutes.", flush=True)
    run_logged(
        docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "up", "-d", "--build"),
        DEPLOY_LOG,
        "Docker Compose deployment failed.",
    )

print("Deployment started. Full Docker output was saved to logs/deploy_hosted_nims.log.")

In [ ]:
ps = subprocess.run(
    docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "ps"),
    text=True,
    capture_output=True,
)
print(ps.stdout)
if ps.returncode != 0:
    print(ps.stderr)

print("\nOpen the sample UI on port 3001 once services are healthy.")

## Prepare data for ingestion

Download the sample product manuals so `notebooks/ingest_data.ipynb` can ingest PDFs without a missing `data/manuals_pdf` directory.

In [ ]:
from urllib.parse import unquote, urlparse
from urllib.request import urlretrieve

manual_list = REPO_ROOT / "data" / "list_manuals.txt"
manual_dir = REPO_ROOT / "data" / "manuals_pdf"
manual_dir.mkdir(parents=True, exist_ok=True)

downloaded = []
skipped = []

for url in manual_list.read_text(encoding="utf-8").splitlines():
    url = url.strip()
    if not url or url.startswith("#"):
        continue

    filename = Path(unquote(urlparse(url).path)).name
    target = manual_dir / filename
    if target.exists() and target.stat().st_size > 0:
        skipped.append(filename)
        continue

    print(f"Downloading {filename}...")
    urlretrieve(url, target)
    downloaded.append(filename)

print(f"Manuals ready in {manual_dir}")
print(f"Downloaded {len(downloaded)} file(s); skipped {len(skipped)} existing file(s).")
print("Next: open notebooks/ingest_data.ipynb to load the sample structured and unstructured data.")